
# CARLA Positioning and Route Exploration

This notebook covers the same learning goals as the reference notebook, but uses a different workflow and code:

- connect to a running CARLA server
- inspect spawn points and actor transforms
- reposition vehicles with custom transforms
- explore map waypoints and road topology
- build and visualize a route between two locations
- move the spectator camera for easier observation

Before running, make sure the CARLA simulator is open and the Python API is available.


In [ ]:

import os
import sys
import time
import random

import carla


In [ ]:

# Optional: add the CARLA PythonAPI folder if needed.
# Update this path for your local installation if imports from agents fail later.
carla_pythonapi = os.environ.get("CARLA_PYTHONAPI", "C:/CARLA_0.9.14/PythonAPI/carla")
if carla_pythonapi not in sys.path:
    sys.path.append(carla_pythonapi)

from agents.navigation.global_route_planner import GlobalRoutePlanner


In [ ]:

client = carla.Client('localhost', 2000)
client.set_timeout(10.0)

world = client.get_world()
carla_map = world.get_map()
spawn_points = carla_map.get_spawn_points()
blueprints = world.get_blueprint_library()

print(f"Map: {carla_map.name}")
print(f"Available spawn points: {len(spawn_points)}")


In [ ]:

# Cleanup helper so the notebook can be re-run safely.
def destroy_actors(patterns=("*vehicle*", "*sensor*")):
    for pattern in patterns:
        for actor in world.get_actors().filter(pattern):
            actor.destroy()

destroy_actors()
print("Old vehicles and sensors removed.")


In [ ]:

# Pick two spawn points and place a vehicle at the first one.
start_transform = random.choice(spawn_points)
goal_transform = random.choice([sp for sp in spawn_points if sp.location.distance(start_transform.location) > 80])

vehicle_bp = random.choice(blueprints.filter('vehicle.tesla.*'))
vehicle = world.try_spawn_actor(vehicle_bp, start_transform)

print("Vehicle blueprint:", vehicle_bp.id)
print("Start transform:")
print(start_transform)
print("Current vehicle transform:")
print(vehicle.get_transform())
print("Goal transform:")
print(goal_transform)



## 1. Compare a blueprint spawn transform with the actor's live transform

The actor usually starts very close to the requested transform, but the simulator may adjust the exact position slightly.


In [ ]:

requested = start_transform
actual = vehicle.get_transform()

location_error = actual.location.distance(requested.location)
yaw_error = abs(actual.rotation.yaw - requested.rotation.yaw)

print(f"Location difference: {location_error:.3f} m")
print(f"Yaw difference: {yaw_error:.3f} deg")



## 2. Reposition the same actor manually

Instead of enabling autopilot first, this notebook explicitly teleports the vehicle to a nearby custom transform.


In [ ]:

shifted_transform = carla.Transform(
    start_transform.location + carla.Location(x=8.0, y=-3.0, z=0.5),
    carla.Rotation(
        pitch=start_transform.rotation.pitch,
        yaw=start_transform.rotation.yaw + 25,
        roll=start_transform.rotation.roll,
    ),
)

vehicle.set_transform(shifted_transform)
time.sleep(0.5)
print(vehicle.get_transform())


In [ ]:

# Put the spectator in an overhead trailing view.
spectator = world.get_spectator()
focus = vehicle.get_transform()

spectator_transform = carla.Transform(
    focus.location + carla.Location(x=-18, y=0, z=12),
    carla.Rotation(pitch=-28, yaw=focus.rotation.yaw, roll=0),
)

spectator.set_transform(spectator_transform)
print("Spectator moved.")



## 3. Inspect waypoints and topology

CARLA maps are made of road segments represented by waypoints. The topology returns pairs of waypoints that describe road connectivity.


In [ ]:

road_topology = carla_map.get_topology()
print(f"Topology pairs: {len(road_topology)}")

entry_wp, exit_wp = road_topology[0]
print("Entry waypoint:", entry_wp)
print("Exit waypoint:", exit_wp)

nearest_wp = carla_map.get_waypoint(vehicle.get_location(), project_to_road=True)
print("Nearest waypoint to vehicle:")
print(nearest_wp)



## 4. Plan a route between two distant points

This uses CARLA's `GlobalRoutePlanner` rather than manually selecting topology pairs.


In [ ]:

planner = GlobalRoutePlanner(carla_map, 2.0)
route = planner.trace_route(start_transform.location, goal_transform.location)

print(f"Route waypoints: {len(route)}")
print("First 5 route steps:")
for waypoint, road_option in route[:5]:
    print(road_option, waypoint.transform.location)


In [ ]:

# Draw the route in the simulator.
for i, (waypoint, road_option) in enumerate(route):
    loc = waypoint.transform.location + carla.Location(z=0.4)
    world.debug.draw_string(
        loc,
        '^',
        draw_shadow=False,
        color=carla.Color(r=0, g=255, b=255),
        life_time=20.0,
        persistent_lines=False,
    )

world.debug.draw_string(
    start_transform.location + carla.Location(z=1.0),
    'START',
    color=carla.Color(r=0, g=255, b=0),
    life_time=20.0,
)
world.debug.draw_string(
    goal_transform.location + carla.Location(z=1.0),
    'GOAL',
    color=carla.Color(r=255, g=0, b=0),
    life_time=20.0,
)

print("Route markers drawn for 20 seconds.")



## 5. Spawn a second vehicle at a dramatic vertical offset

The original notebook experimented with dropping cars from the sky. This version does the same idea with a helper transform and a different actor.


In [ ]:

second_bp = random.choice(blueprints.filter('vehicle.audi.*') or blueprints.filter('vehicle.*'))
second_spawn = random.choice(spawn_points)
second_vehicle = world.try_spawn_actor(second_bp, second_spawn)

sky_transform = carla.Transform(
    vehicle.get_transform().location + carla.Location(x=4, y=2, z=8),
    carla.Rotation(yaw=vehicle.get_transform().rotation.yaw),
)

second_vehicle.set_transform(sky_transform)
print("Second vehicle:", second_bp.id)
print(second_vehicle.get_transform())


In [ ]:

# Optional: let the first vehicle drive while you watch the scene.
vehicle.set_autopilot(True)
print("Autopilot enabled for the first vehicle.")



## 6. Final cleanup

Run this when you are done experimenting.


In [ ]:

destroy_actors()
print("Scene cleaned up.")
